# Biocomputing with Brainoware
### Procedure 1 - Human cortical organoid generation and Brainoware hardware integration
#### 1. Spike sorting with Kilosort2 and functional connectivity calculation (local version)
**Author: Hongwei Cai, Huiyu Chu**  
**Date: June 6, 2026**  
**Description**: This notebook provides a step-by-step guide for spike sorting using Kilosort2 in local environment, specifically based on "Network Scan" result of cortical organoid attached on the MaxOne MEA. It covers dependency installation, raw data processing, sorting, quality metric computation, curation, and functional connectivity analysis.


In [ ]:
import glob
import re
import os
import matplotlib.pyplot as plt
import numpy as np  
import pandas as pd

import spikeinterface as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.comparison as sc
import spikeinterface.widgets as sw
from spikeinterface.postprocessing import compute_unit_locations
from spikeinterface import extract_waveforms
from matplotlib.pyplot import figure
import networkx as nx

%matplotlib inline
# encoding: utf-8

In [ ]:
# Set Kilosort2 path
ss.Kilosort2Sorter.set_kilosort2_path('D://Kilosort-2.0')

# Define Kilosort2 parameters
kilo_params = ss.Kilosort2Sorter.default_params()
kilo_params['minFR'] = 0.02
kilo_params['minfr_goodchannels'] = 0.1
kilo_params['detect_threshold'] = 5.5
kilo_params['n_jobs'] = 8
kilo_params

In [ ]:


# Base directory
base_path = "./"

# Find all data.raw.h5 files under Network/<number>/
pattern = os.path.join(base_path, 'fc*.raw.h5')
all_files = glob.glob(pattern, recursive=True)

# Loop through files
for file_path in all_files:
    folder = os.path.dirname(file_path)
    sorter_output = os.path.join(folder, 'kilosort2')
    
    if os.path.exists(sorter_output):
        print(f"Skipping {folder} — 'kilosort2' already exists.")
        continue

    try:
        print(f"Processing: {file_path}")
        
        # Load recording
        recording = se.MaxwellRecordingExtractor(file_path)
        recording_signed = spre.unsigned_to_signed(recording)
        # Apply bandpass filter
        recording_filtered = spre.bandpass_filter(recording_signed, freq_min=300, freq_max=6000)
        
        # Run sorter
        sorting = ss.run_sorter(
            sorter_name="kilosort2",
            recording=recording_filtered,
            folder=sorter_output,
            verbose=True,
            **kilo_params
        )
        
        waveform_extractor = extract_waveforms(
            recording=recording_filtered,
            sorting=sorting,
            folder=os.path.join(sorter_output, "waveforms"),
            ms_before=1.0,
            ms_after=2.0,
            max_spikes_per_unit=500,
            overwrite=None,   # not True/False
            n_jobs=8
        )
        # Compute unit locations (default method is 'center_of_mass')
        compute_unit_locations(waveform_extractor)

        # Get the computed locations
        unit_locations = waveform_extractor.load_extension("unit_locations").get_data()

        unit_ids = sorting.unit_ids
        duration_s = recording.get_total_duration()
        
        
        firing_rates = []
        for unit_id in unit_ids:
            spike_train = sorting.get_unit_spike_train(unit_id)
            firing_rate = len(spike_train) / duration_s
            firing_rates.append(firing_rate)

        df_firing = pd.DataFrame({
            "unit_id": unit_ids,
            "firing_rate": firing_rates
        })

        # -----------------------------
        # Spike trains table
        # -----------------------------
        rows = []
        for unit_id in unit_ids:
            spike_times = sorting.get_unit_spike_train(unit_id)
            for t in spike_times:
                rows.append({"unit_id": unit_id, "time": t})

        df_spikes = pd.DataFrame(rows)

        # -----------------------------
        # Unit locations table
        # -----------------------------
        xy_locations = unit_locations[:, :2]
        df_locations = pd.DataFrame(xy_locations, columns=["x", "y"])
        df_locations.insert(0, "unit_id", unit_ids)

        df_merged = pd.merge(df_locations, df_firing, on="unit_id")

        # -----------------------------
        # Plot
        # -----------------------------
        ax1 = df_merged.plot.scatter(
            x="x",
            y="y",
            s=df_merged["firing_rate"] * 70,
            c="unit_id",
            colormap="cividis",
            alpha=0.8
        )
        ax1.set_aspect("equal")
        ax1.figure.set_size_inches(12, 9)
        ax1.set_ylabel("Y (μm)")
        ax1.set_xlabel("X (μm)")

        plt.savefig(os.path.join(sorter_output, "firing_map.svg"), bbox_inches="tight")
        plt.savefig(os.path.join(sorter_output, "firing_map.png"), bbox_inches="tight")
        plt.close()

        # -----------------------------
        # Save outputs
        # -----------------------------
        df_spikes.to_csv(os.path.join(sorter_output, "spike_trains.csv"), index=False)
        df_merged[["unit_id", "x", "y"]].to_csv(
            os.path.join(sorter_output, "neuron_coord.csv"),
            index=False
        )

        print(f"Done: {folder}")

        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")


# Functional connectivity from STTC

Compute the spike-time tiling coefficient (STTC) directly from the sorting result `df_spikes` (columns `unit_id`, `time` in samples), then draw a functional connectivity map whose node positions come from `df_locations` (columns `unit_id`, `x`, `y`).


In [ ]:
# Define STTC matrix computation using Elephant (Cutts & Eglen 2014)
import numpy as np
import neo
import quantities as pq
from elephant.spike_train_correlation import spike_time_tiling_coefficient


def compute_sttc_matrix(spike_trains_s, dt_s, t_start_s, t_stop_s):
    """Symmetric STTC matrix using elephant.spike_time_tiling_coefficient.

    Parameters
    ----------
    spike_trains_s : list of 1D arrays
        Spike times in seconds, one array per unit.
    dt_s : float
        Synchrony window (+/- dt) in seconds.
    t_start_s, t_stop_s : float
        Recording start/stop in seconds (used for the tiling fraction TA/TB).

    Returns
    -------
    np.ndarray (n x n)
        STTC values; diagonal left as NaN (no self-connections), NaN for empty trains.
    """
    n = len(spike_trains_s)
    neo_trains = [
        neo.SpikeTrain(np.sort(np.asarray(st, dtype=float)) * pq.s,
                       t_start=t_start_s * pq.s, t_stop=t_stop_s * pq.s)
        for st in spike_trains_s
    ]
    dt_q = dt_s * pq.s
    sttc_matrix = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(i + 1, n):
            sttc = spike_time_tiling_coefficient(neo_trains[i], neo_trains[j], dt=dt_q)
            sttc_matrix[i, j] = sttc
            sttc_matrix[j, i] = sttc
    return sttc_matrix


In [ ]:
# Compute STTC from the in-memory sorting result `df_spikes` (Elephant)
import pandas as pd

# Settings
sampling_rate = 20000          # Hz
dt = 0.030                     # 30 ms coincidence window (seconds)
recording_duration_s = None    # set to the recording length in seconds if known;
                               # otherwise the last spike time is used as t_stop

# Per-unit spike trains (sorted, in seconds), keyed by unit_id
spike_trains = {
    unit: np.sort(group["time"].values.astype(np.float64)) / sampling_rate
    for unit, group in df_spikes.groupby("unit_id")
}

# Fixed unit order used for every matrix / coordinate downstream
selected_unit_ids = sorted(spike_trains.keys())
n_units = len(selected_unit_ids)

# List of spike-train arrays (same order as selected_unit_ids)
spike_trains_list = [spike_trains[uid] for uid in selected_unit_ids]

# Recording span in seconds (t_stop must be >= the last spike)
last_spike_s = float(df_spikes["time"].max()) / sampling_rate
duration = max(recording_duration_s or 0.0, last_spike_s)

# Run STTC (elephant.spike_time_tiling_coefficient for every unit pair)
sttc_fast = compute_sttc_matrix(spike_trains_list, dt, 0.0, duration)

# Labelled DataFrame (rows/cols are unit_id)
sttc_df = pd.DataFrame(sttc_fast, index=selected_unit_ids, columns=selected_unit_ids)
sttc_df


## Build the thresholded adjacency matrix


In [ ]:
# Plot / threshold parameters
color_plot = 5
color_shift = 3
color_total = color_plot + color_shift
width_total = 5
dw = 1
thred_c = 0.3        # adjacency cutoff: STTC below this -> no edge
thred_delta = 0.25
thred_plot = 0.35    # only edges above this are drawn
color_map = 'RdYlGn_r'
node_color = 'mediumseagreen'


In [ ]:
# Adjacency matrix: replace NaN with 0 and drop weak / negative couplings
adj_matrix = np.nan_to_num(sttc_fast, nan=0.0)
adj_matrix[adj_matrix < thred_c] = 0


## Align unit locations to the STTC unit order


In [ ]:
# Map each STTC unit to its (x, y) from `df_locations`, in selected_unit_ids order.
# This guarantees node i of the graph sits at the location of selected_unit_ids[i].
coord_lookup = df_locations.set_index("unit_id")[["x", "y"]]
np_coord = coord_lookup.loc[selected_unit_ids].to_numpy()
np_coord


In [ ]:
# Build the graph; node index i corresponds to selected_unit_ids[i]
nx_adj = nx.from_numpy_array(adj_matrix)
pos_ = np_coord                       # use real electrode coordinates as layout
sorted_edges = sorted(nx_adj.edges(data=True), key=lambda x: x[2]['weight'])


## Draw the functional connectivity map


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mc
import plotly.graph_objects as go


In [ ]:
cmap = plt.cm.get_cmap(color_map, color_total + 4)


In [ ]:
widths = np.arange(0, dw * width_total, dw) + 1
for i in range(0, width_total):
    widths[i] = (widths[i] / 2) ** 2 + 1

colors_with_alpha = []
for i in range(color_total):
    rgba = list(cmap(i))
    alpha = (i + 1) / color_total      # ramp transparency with strength
    rgba[3] = alpha
    colors_with_alpha.append(tuple(rgba))


def rgba_to_plotly_string(rgba_tuple):
    r, g, b, a = rgba_tuple
    return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a:.3f})"


color_scale = [rgba_to_plotly_string(rgba) for rgba in colors_with_alpha]


In [ ]:
def make_edge(x, y, text, width, colors):
    """Scatter trace for a single edge between two nodes."""
    return go.Scatter(x=x,
                      y=y,
                      line=dict(width=width, color=colors),
                      hoverinfo='text',
                      text=([text]),
                      mode='lines')


In [ ]:
# Build edge traces (strongest-on-top via sorted_edges)
edge_trace = []
for edge in sorted_edges:
    if edge[2]['weight'] > thred_plot:
        char_1 = edge[0]
        char_2 = edge[1]

        x0, y0 = pos_[char_1]
        x1, y1 = pos_[char_2]

        text = None

        if edge[2]['weight'] < 1:
            color_num = int((edge[2]['weight'] - thred_plot) / (1 - thred_plot) * color_plot - 1) + 2
            width_num = int((edge[2]['weight'] - thred_plot) / (1 - thred_plot) * width_total - 1)
        else:
            color_num = int(color_plot - 1) + 2
            width_num = int(width_total - 1)

        trace = make_edge([x0, x1, None], [y0, y1, None], text,
                          widths[width_num], color_scale[color_num])
        edge_trace.append(trace)

# Node trace: position from df_locations, size from total connectivity, label = unit_id
node_trace = go.Scatter(x=[], y=[], text=[],
                        textposition="top center",
                        textfont_size=10,
                        mode='markers+text',
                        hoverinfo='text',
                        marker=dict(color=[], size=[], line=None))

for node in nx_adj.nodes():
    x, y = pos_[node]
    node_trace['x'] += tuple([x])
    node_trace['y'] += tuple([y])
    node_trace['marker']['color'] += tuple([node_color])
    node_trace['marker']['size'] += tuple([1 * (sum(adj_matrix[:, node])) ** (1 / 3)])
    #node_trace['text'] += tuple(['<b>' + str(selected_unit_ids[node]) + '</b>'])

layout = go.Layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    width=1000,
    height=800
)

fig = go.Figure(layout=layout)
for trace in edge_trace:
    fig.add_trace(trace)
fig.add_trace(node_trace)

fig.update_layout(showlegend=False)
fig.update_xaxes(showticklabels=True)
fig.update_yaxes(showticklabels=True, scaleanchor="x", scaleratio=1)

fig.show()
